In [ ]:
from pathlib import Path
from functools import reduce

import requests
import pandas as pd

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

ECOS_API_KEY = os.getenv("ECOS_API_KEY")

In [ ]:
# 프로젝트 경로 설정
BASE_DIR = Path.cwd().parent

# ECOS API 설정
BASE_URL = "https://ecos.bok.or.kr/api/StatisticSearch"

START_DATE_M = "200001"
END_DATE_M = "202606"

START_DATE_Q = "2000Q1"
END_DATE_Q = "2026Q2"

In [32]:
ECOS_SERIES = [
    {"name": "기준금리", "column": "base_rate", "stat_code": "722Y001", "item_codes": "0101000", "cycle": "M"},
    {"name": "실업률", "column": "unemployment", "stat_code": "901Y027", "item_codes": ["I61BC", "I28B"], "cycle": "M"},
    {"name": "경기동행종합", "column": "coincident", "stat_code": "901Y067", "item_codes": ["I16B"], "cycle": "M"},
    {"name": "경기선행종합", "column": "leading_price", "stat_code": "901Y067", "item_codes": ["I16A"], "cycle": "M"},
    {"name": "경제심리지수(ESI)", "column": "ESI", "stat_code": "513Y001", "item_codes": "E2000", "cycle": "M"},
    {"name": "수출물가지수", "column": "export_price", "stat_code": "402Y014", "item_codes": ["*AA", "W"], "cycle": "M"},
    {"name": "제조업가동률", "column": "manufacturing_capacity_utilization", "stat_code": "901Y035", "item_codes": ["I32A", "I11C"], "cycle": "M"},
    {"name": "경제성장률", "column": "economic_growth_rate", "stat_code": "902Y015", "item_codes": ["KOR"], "cycle": "Q"},
    {"name": "소비자심리지수(CCSI)", "column": "CCSI", "stat_code": "511Y002", "item_codes": ["FME", "99988"], "cycle": "M"},
    {"name": "미국 산업생산", "column": "us_industrial_production", "stat_code": "902Y020", "item_codes": ["USA"], "cycle": "M"},
    {"name": "코스닥지수", "column": "kosdaq_index", "stat_code": "901Y014", "item_codes": ["2090000"], "cycle": "M"},
    {"name": "자본수지", "column": "capital_account", "stat_code": "301Y013", "item_codes": ["BOPC00000000"], "cycle": "M"},
    {"name": "미국 정책금리", "column": "us_policy_rate", "stat_code": "902Y023", "item_codes": ["IR3TIB", "USA"], "cycle": "M"},
    {"name": "지가변동률", "column": "land_price_change", "stat_code": "901Y064", "item_codes": ["P65A"], "cycle": "M"},
    {"name": "국내총생산(GDP)", "column": "GDP", "stat_code": "200Y102", "item_codes": ["10111"], "cycle": "Q"},
    {"name": "경상수지", "column": "current_account", "stat_code": "301Y017", "item_codes": ["SA000"], "cycle": "M"},
    {"name": "미국 소비자물가지수", "column": "us_CPI", "stat_code": "902Y008", "item_codes": ["US"], "cycle": "M"},
    # ===== 아직 ECOS_SERIES에 추가되지 않은 변수 =====
    {"name": "취업자수", "column": "employment", "stat_code": "901Y027", "item_codes": ["I61BA", "I28A"], "cycle": "M"},
    {"name": "기업경기실사지수(BSI)", "column": "business_survey", "stat_code": "512Y007", "item_codes": ["AA", "99988"], "cycle": "M"},
    {"name": "전산업생산", "column": "industrial_production", "stat_code": "901Y033", "item_codes": ["A00", "2"], "cycle": "M"},
    {"name": "설비투자", "column": "capital_investment", "stat_code": "901Y066", "item_codes": ["I15B"], "cycle": "M"},
    {"name": "소비자물가지수", "column": "CPI", "stat_code": "901Y009", "item_codes": ["0"], "cycle": "M"},
    {"name": "본원통화", "column": "monetary_base", "stat_code": "102Y004", "item_codes": ["ABA1"], "cycle": "M"},
    {"name": "광의통화(M2)", "column": "broad_money", "stat_code": "161Y005", "item_codes": ["BBHS00"], "cycle": "M"},
    {"name": "순상품교역조건", "column": "terms_of_trade", "stat_code": "403Y005", "item_codes": ["A"], "cycle": "M"},
    {"name": "수출금액", "column": "export_value", "stat_code": "403Y001", "item_codes": ["*AA"], "cycle": "M"},
    {"name": "소매판매액", "column": "retail_sales", "stat_code": "901Y100", "item_codes": ["G0", "T3"], "cycle": "Q"},
    {"name": "경제활동참가율", "column": "labor_force_participation_rate", "stat_code": "901Y027", "item_codes": ["I61D", "I28B"], "cycle": "M"},
    {"name": "신용카드 이용금액", "column": "credit_card_spending", "stat_code": "601Y003", "item_codes": ["200000", "00"], "cycle": "M"},
    {"name": "광공업생산", "column": "manufacturing_production", "stat_code": "901Y032", "item_codes": ["I11AA", "2"], "cycle": "M"},
    {"name": "제조업생산", "column": "manufacturing_output", "stat_code": "901Y032", "item_codes": ["I11AC", "2"], "cycle": "M"},
    {"name": "서비스업생산", "column": "service_production", "stat_code": "901Y038", "item_codes": ["I51A", "3"], "cycle": "M"},
    {"name": "생산자물가지수", "column": "PPI", "stat_code": "404Y014", "item_codes": ["*AA"], "cycle": "M"},
    {"name": "금융기관유동성(Lf)", "column": "financial_institution_liquidity", "stat_code": "171Y003", "item_codes": ["LAS0000"], "cycle": "M"},
    {"name": "광의유동성(L)", "column": "broad_liquidity", "stat_code": "172Y001", "item_codes": ["XS00000"], "cycle": "M"},
    {"name": "예금은행 수신금리", "column": "deposit_rate", "stat_code": "121Y002", "item_codes": ["BEABAA2"], "cycle": "M"},
    {"name": "예금은행 대출금리", "column": "loan_rate", "stat_code": "121Y006", "item_codes": ["BECBLA01"], "cycle": "M"},
    {"name": "예금은행 총예금", "column": "total_deposits", "stat_code": "104Y015", "item_codes": ["BDAA1"], "cycle": "M"},
    {"name": "예금은행 대출금", "column": "loans_outstanding", "stat_code": "104Y016", "item_codes": ["BDCA1"], "cycle": "M"},
    {"name": "수입물가지수", "column": "import_price", "stat_code": "401Y015", "item_codes": ["*AA", "W"], "cycle": "M"},
    {"name": "소득교역조건지수", "column": "terms_of_income_trade", "stat_code": "403Y005", "item_codes": ["B"], "cycle": "M"},
    {"name": "수입금액", "column": "import_value", "stat_code": "403Y003", "item_codes": ["*AA"], "cycle": "M"},
    {"name": "금융계정", "column": "financial_account", "stat_code": "301Y013", "item_codes": ["BOPF00000000"], "cycle": "M"},
    {"name": "가계신용", "column": "household_credit", "stat_code": "151Y001", "item_codes": ["1000000"], "cycle": "Q"},
]

In [33]:
def fetch_ecos_series(stat_code, item_codes, column_name, cycle="M"):

    # item_codes를 리스트로 통일
    if isinstance(item_codes, str):
        item_codes = [item_codes]

    # 주기별 조회 기간 설정
    if cycle == "Q":
        start_date = START_DATE_Q
        end_date = END_DATE_Q
    else:
        start_date = START_DATE_M
        end_date = END_DATE_M

    # API URL 생성
    url = (
        f"{BASE_URL}/{ECOS_API_KEY}/json/kr/1/1000/"
        f"{stat_code}/{cycle}/{start_date}/{end_date}/"
        + "/".join(item_codes)
    )

    # API 요청
    response = requests.get(url)
    response.raise_for_status()
    result = response.json()

    # 데이터 존재 여부 확인
    if "StatisticSearch" not in result:
        print(f"수집 실패: {column_name}")
        print(result)
        return None

    df = pd.DataFrame(result["StatisticSearch"]["row"])[["TIME", "DATA_VALUE"]]
    df.columns = ["date", column_name]
    
    df[column_name] = pd.to_numeric(df[column_name], errors="coerce")
    
    # 분기(Q) 데이터를 월별로 확장
    if cycle == "Q":
    
        monthly_rows = []
    
        for _, row in df.iterrows():
    
            year = row["date"][:4]
            quarter = row["date"][-2:]
    
            if quarter == "Q1":
                months = ["01", "02", "03"]
            elif quarter == "Q2":
                months = ["04", "05", "06"]
            elif quarter == "Q3":
                months = ["07", "08", "09"]
            else:
                months = ["10", "11", "12"]
    
            for month in months:
                monthly_rows.append({
                    "date": f"{year}{month}",
                    column_name: row[column_name]
                })
    
        df = pd.DataFrame(monthly_rows)
    
    return df

In [34]:
dataframes = []

for series in ECOS_SERIES:
    df = fetch_ecos_series(
        stat_code=series["stat_code"],
        item_codes=series["item_codes"],
        column_name=series["column"],
        cycle=series["cycle"]
    )

    if df is not None:
        dataframes.append(df)
        print(f"수집 완료: {series['name']} → {series['column']}")

dataframes

수집 완료: 기준금리 → base_rate
수집 완료: 실업률 → unemployment
수집 완료: 경기동행종합 → coincident
수집 완료: 경기선행종합 → leading_price
수집 완료: 경제심리지수(ESI) → ESI
수집 완료: 수출물가지수 → export_price
수집 완료: 제조업가동률 → manufacturing_capacity_utilization
수집 완료: 경제성장률 → economic_growth_rate
수집 완료: 소비자심리지수(CCSI) → CCSI
수집 완료: 미국 산업생산 → us_industrial_production
수집 완료: 코스닥지수 → kosdaq_index
수집 완료: 자본수지 → capital_account
수집 완료: 미국 정책금리 → us_policy_rate
수집 완료: 지가변동률 → land_price_change
수집 완료: 국내총생산(GDP) → GDP
수집 완료: 경상수지 → current_account
수집 완료: 미국 소비자물가지수 → us_CPI
수집 완료: 취업자수 → employment
수집 완료: 기업경기실사지수(BSI) → business_survey
수집 완료: 전산업생산 → industrial_production
수집 완료: 설비투자 → capital_investment
수집 완료: 소비자물가지수 → CPI
수집 완료: 본원통화 → monetary_base
수집 완료: 광의통화(M2) → broad_money
수집 완료: 순상품교역조건 → terms_of_trade
수집 완료: 수출금액 → export_value
수집 완료: 소매판매액 → retail_sales
수집 완료: 경제활동참가율 → labor_force_participation_rate
수집 완료: 신용카드 이용금액 → credit_card_spending
수집 완료: 광공업생산 → manufacturing_production
수집 완료: 제조업생산 → manufacturing_output
수집 완료: 서비스업생산 

[       date  base_rate
 0    200001       4.75
 1    200002       5.00
 2    200003       5.00
 3    200004       5.00
 4    200005       5.00
 ..      ...        ...
 313  202602       2.50
 314  202603       2.50
 315  202604       2.50
 316  202605       2.50
 317  202606       2.50
 
 [318 rows x 2 columns],
        date  unemployment
 0    200001           5.0
 1    200002           4.8
 2    200003           4.5
 3    200004           4.5
 4    200005           4.4
 ..      ...           ...
 312  202601           3.0
 313  202602           2.9
 314  202603           2.7
 315  202604           2.8
 316  202605           2.8
 
 [317 rows x 2 columns],
        date  coincident
 0    200001        46.5
 1    200002        46.7
 2    200003        47.1
 3    200004        47.5
 4    200005        48.0
 ..      ...         ...
 312  202601       114.7
 313  202602       115.5
 314  202603       116.2
 315  202604       116.5
 316  202605       116.3
 
 [317 rows x 2 columns],
       

In [37]:
# df 만들기
from functools import reduce

master_df = reduce(
    lambda left, right: pd.merge(left, right, on="date", how="outer"),
    dataframes
)

master_df = (
    master_df
    .sort_values("date")
    .reset_index(drop=True)
)

print(master_df.shape)
master_df.head()

(318, 45)


,date,base_rate,unemployment,coincident,leading_price,ESI,export_price,manufacturing_capacity_utilization,economic_growth_rate,CCSI,...,broad_liquidity,deposit_rate,loan_rate,total_deposits,loans_outstanding,import_price,terms_of_income_trade,import_value,financial_account,household_credit
0,200001,4.75,5.0,46.5,50.3,NaN,141.31,108.883,1.995,NaN,...,NaN,7.15,8.59,326441.0,256737.6,75.71,37.74,32.57,-734.3,NaN
1,200002,5.00,4.8,46.7,50.5,NaN,137.81,108.079,1.995,NaN,...,NaN,7.24,8.64,343188.4,259581.8,76.03,37.17,30.85,683.0,NaN
2,200003,5.00,4.5,47.1,50.7,NaN,135.30,109.169,1.995,NaN,...,NaN,7.24,8.79,347710.3,266964.5,75.99,42.37,36.48,-402.7,NaN
3,200004,5.00,4.5,47.5,50.6,NaN,133.88,107.281,1.541,NaN,...,NaN,7.19,8.61,357049.8,277224.1,73.99,39.42,34.50,-45.3,NaN
4,200005,5.00,4.4,48.0,50.7,NaN,134.99,112.181,1.541,NaN,...,NaN,7.06,8.62,363953.3,285120.7,75.57,43.70,34.14,789.8,NaN


In [38]:
# 저장
from pathlib import Path

PROCESSED_DATA_DIR = Path("data/processed")

save_path = PROCESSED_DATA_DIR / "master_df.csv"

master_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"저장 완료: {save_path}")

저장 완료: data\processed\master_df.csv
